# 02 · Copy Dataset from Drive & Test DataLoaders

**This notebook:**
1. Clones repo & mounts Drive
2. Copies your pre-split dataset from Drive → Colab SSD (fast NVMe)
3. Tests the DataLoaders

> No editing needed — just run all cells top-to-bottom.

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────
import os, sys

GITHUB_USER = 'musarashid49'
REPO_NAME   = 'Image-Classification-with-CNN'
REPO_DIR    = f'/content/{REPO_NAME}'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')
print(f'✓ Ready. Working dir: {os.getcwd()}')

In [ ]:
# ── Cell 2: Define paths ─────────────────────────────────────────────
DRIVE_DATASET_PATH  = '/content/drive/MyDrive/dataset'
LOCAL_DATA_DIR      = '/content/data'
DRIVE_RESULTS_PATH  = '/content/drive/MyDrive/pk_politicians_results'

from pathlib import Path
assert Path(DRIVE_DATASET_PATH).is_dir(), f'Not found: {DRIVE_DATASET_PATH}'
print(f'✓ Drive dataset exists: {DRIVE_DATASET_PATH}')

In [ ]:
# ── Cell 3: Copy dataset from Drive → Colab SSD (fast) ───────────────
import shutil, time

if os.path.isdir(LOCAL_DATA_DIR) and len(os.listdir(LOCAL_DATA_DIR)) > 0:
    print(f'Dataset already copied to {LOCAL_DATA_DIR} — skipping copy.')
else:
    print(f'Copying from Drive to local SSD (this takes ~2-5 min)...')
    t0 = time.time()
    shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATA_DIR)
    elapsed = time.time() - t0
    print(f'✓ Done in {elapsed:.1f}s')

In [ ]:
# ── Cell 4: Test DataLoaders ─────────────────────────────────────────
import torch
from src.dataset import get_dataloaders
from src.utils import set_seed

set_seed(42)
loaders = get_dataloaders(
    train_dir = os.path.join(LOCAL_DATA_DIR, 'train'),
    val_dir   = os.path.join(LOCAL_DATA_DIR, 'val'),
    test_dir  = os.path.join(LOCAL_DATA_DIR, 'test'),
)

# Sample a batch
imgs, labels = next(iter(loaders['train']))
print(f'\nSample batch:')
print(f'  images shape: {imgs.shape}  (batch_size, channels, height, width)')
print(f'  labels shape: {labels.shape}')
print(f'  pixel range after normalization: [{imgs.min():.3f}, {imgs.max():.3f}]')
print(f'\n✓ DataLoaders ready for training!')
print('\nNext → open 03_train_resnet50.ipynb')